# 12 Fallback on failure: every tier ends in a typed outcome

A hosted language model times out, hits a rate limit, or returns a label outside the route list. In a system built around that model, the customer's message gets an exception. In the cascade from notebook 05 the same failure happens on only the 8.5% of messages that reach the slow tier, and a rule written in advance says where those messages go.

This notebook injects failures into the Meridian test run and compares four designs: a language model alone, the same with one retry, a cascade with no failure rule, and a cascade whose slow tier falls back to a person. It then asks what the cascade should do when the encoder itself is down, and checks the answer against the release checklist from notebook 06.

**What the numbers cover.** 200 test messages. Failures are simulated: a fixed hash of the message id decides which calls fail, so every run gives the same result. They are not observed outages, and a real outage is correlated in time (every call in a minute fails), which this does not model. Each answer comes from cached embeddings and the recorded slow-tier file, so the notebook runs in seconds.

In [1]:
import json, hashlib
import numpy as np
from pathlib import Path
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
import meridian as m

RESULTS = Path("data/results")
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
r03 = json.loads((RESULTS / "03_calibration_and_conformal.json").read_text())
ENC, T, ALPHA = r03["encoder"], r03["T"], r03["alpha_chosen"]
Q = r03[f"conformal_alpha_{ALPHA}"]["q"]

tr, te = m.load("train"), m.load("test")
_, ytr = m.texts_labels(tr); _, yte = m.texts_labels(te)
E = m.cached_embeddings(ENC)
clf = LogisticRegression(C=r01[ENC]["C"], max_iter=3000).fit(E["train"], ytr)
P = softmax(clf.decision_function(E["test"]) / T, axis=1)
route_set = [set(np.where(p >= 1 - Q)[0]) for p in P]          # the conformal gate from notebook 03
idmap = json.loads(Path("data/llm/test_id_map.json").read_text())
ans = {idmap[j["id"]]: j["route"] for j in map(json.loads, Path("data/llm/test_haiku_answers.jsonl").read_text().splitlines())}
slow = np.array([m.ROUTE_ID[ans[r["id"]]] for r in te])        # recorded slow-tier answer for every message

n = len(te); one = np.array([len(s) == 1 for s in route_set])
print(f"{n} messages | the gate acts on {one.sum()} ({one.mean():.1%}) and escalates {(~one).sum()} ({(~one).mean():.1%})")

200 messages | the gate acts on 183 (91.5%) and escalates 17 (8.5%)


**Goal.** Fix the starting point: the gate decides alone on 183 messages, and only the other 17 ever need the slow tier. That split is what limits how far an outage of the slow tier can reach.

## Part 1. When the slow tier fails

`fails(message, tier, attempt, p)` returns True when a simulated call fails: it hashes the message id, the tier and the attempt number into a number between 0 and 1 and compares it with the failure rate `p`. The same message at the same rate always gives the same answer, so a test can assert on it.

Four designs handle a failed call differently:

- **LLM alone, no fallback.** Every message goes to the language model. A failed call returns nothing.
- **LLM alone, one retry.** The same, with one second attempt.
- **Cascade, no failure rule.** The encoder decides when the gate is sure; an escalated message whose slow-tier call fails returns nothing.
- **Cascade with fallback.** An escalated message whose slow-tier call fails goes to a person, so the outcome is typed: `encoder`, `llm` or `person`.

In [2]:
def fails(i, tier, attempt, p):
    h = hashlib.sha256(f"{i}|{tier}|{attempt}".encode()).digest()
    return int.from_bytes(h[:8], "big") / 2**64 < p

def llm_alone(i, p, retries=0):
    for a in range(retries + 1):
        if not fails(i, "slow", a, p): return ("llm", int(slow[i]))
    return (None, None)                                  # no typed outcome: the caller gets an error

def cascade(i, p, fallback):
    if one[i]: return ("encoder", next(iter(route_set[i])))
    if not fails(i, "slow", 0, p): return ("llm", int(slow[i]))
    return ("person", None) if fallback else (None, None)

DESIGNS = {"LLM alone, no fallback": lambda i, p: llm_alone(i, p),
           "LLM alone, one retry": lambda i, p: llm_alone(i, p, retries=1),
           "cascade, no failure rule": lambda i, p: cascade(i, p, False),
           "cascade with fallback": lambda i, p: cascade(i, p, True)}

def summarize(design, p):
    out = [DESIGNS[design](i, p) for i in range(n)]
    who = [o[0] for o in out]
    auto = [i for i, o in enumerate(out) if o[0] in ("encoder", "llm")]
    right = int(sum(out[i][1] == yte[i] for i in auto))
    return {"typed_missing": who.count(None) / n, "person": who.count("person") / n,
            "answered": len(auto) / n, "right": right, "n_auto": len(auto)}

rows = {}
print("messages out of 200: answered by a machine / lost (no typed outcome) / sent to a person")
print(f"{'slow-tier failure rate':>24s}" + "".join(f"{d:>28s}" for d in DESIGNS))
for p in (0.0, 0.1, 0.5, 1.0):
    cells = []
    for d in DESIGNS:
        s = summarize(d, p); rows[(d, p)] = s
        cells.append(f"{s['n_auto']} / {round(s['typed_missing'] * n)} / {round(s['person'] * n)}")
    print(f"{p:>24.0%}" + "".join(f"{c:>28s}" for c in cells))

messages out of 200: answered by a machine / lost (no typed outcome) / sent to a person
  slow-tier failure rate      LLM alone, no fallback        LLM alone, one retry    cascade, no failure rule       cascade with fallback
                      0%                 200 / 0 / 0                 200 / 0 / 0                 200 / 0 / 0                 200 / 0 / 0
                     10%                181 / 19 / 0                 198 / 2 / 0                 200 / 0 / 0                 200 / 0 / 0
                     50%               100 / 100 / 0                154 / 46 / 0                 195 / 5 / 0                 195 / 0 / 5
                    100%                 0 / 200 / 0                 0 / 200 / 0                183 / 17 / 0                183 / 0 / 17


**Goal.** Count, for each design and failure rate, how many of the 200 messages the machine answered, how many ended with no typed outcome at all ("lost": an exception the caller must handle), and how many went to a person. Each cell reads answered / lost / person.

**What we got.** With the slow tier failing on every call, the language model alone answers 0% of messages, and one retry changes nothing because the retry fails too. The cascade still answers 91.5% of messages from the encoder alone. Without a failure rule, the other 17 are lost; with the fallback they go to a person, and no message is without an outcome at any failure rate. At a 10% failure rate the language model alone loses 19 of 200 messages and one retry cuts that to 2; the cascade loses none in this run (an expected 1.7 of its 17 escalated messages would fail, and none happened to). At 50% the cascade loses 5 without a failure rule and none with one. These are simulated failures with no correlation in time, so read them as a design comparison and not as a forecast.

## Part 2. Accuracy of the answers the machine still gives

A fallback must not trade availability for wrong answers. The machine answers fewer messages when the slow tier is down; the question is whether the ones it still answers are as accurate as before.

In [3]:
from math import comb
print(f"{'design, slow tier fully down':34s}{'answered':>10s}{'right of answered':>22s}{'Wilson 95%':>18s}")
for d in ("LLM alone, no fallback", "cascade with fallback"):
    for p in (0.0, 1.0):
        s = summarize(d, p)
        if s["n_auto"] == 0:
            print(f"{d + f' (p={p:.0%})':34s}{0:>10d}{'no answers':>22s}"); continue
        lo, hi = m.wilson(s["right"], s["n_auto"])
        print(f"{d + f' (p={p:.0%})':34s}{s['n_auto']:>10d}{s['right']:>14d} of {s['n_auto']:<4d}[{lo:.3f}, {hi:.3f}]")

design, slow tier fully down        answered     right of answered        Wilson 95%
LLM alone, no fallback (p=0%)            200           190 of 200 [0.910, 0.973]
LLM alone, no fallback (p=100%)            0            no answers
cascade with fallback (p=0%)             200           195 of 200 [0.943, 0.989]
cascade with fallback (p=100%)           183           180 of 183 [0.953, 0.994]


**What we got.** With the slow tier out, the cascade answers 183 messages and is right on 180 (98.4%, Wilson interval 95.3% to 99.4%), the same messages and the same accuracy it had when the slow tier was up, because the gate chose those 183 before the slow tier was involved. The outage costs 17 more messages for a person, not accuracy on the rest.

## Part 3. When the encoder itself is down

The encoder runs on the same machine as the cascade, so its failure is rarer, but it needs a plan too. There are two fallbacks and each breaks a different budget from the release checklist (notebook 06): send every message to the slow tier (cost), or to a person (review capacity).

In [4]:
CHECK = {"calls_per_1000": 120, "person_share": 0.05}      # from the release checklist targets in notebook 06
options = {"encoder up (normal run)": {"calls": (~one).mean() * 1000, "person": 0.0},
           "encoder down, send all to the slow tier": {"calls": 1000.0, "person": 0.0},
           "encoder down, send all to a person": {"calls": 0.0, "person": 1.0}}
print(f"{'':42s}{'slow-tier calls /1000':>24s}{'to a person':>14s}  checklist (<=120 calls, <=5% people)")
for name, o in options.items():
    ok = o["calls"] <= CHECK["calls_per_1000"] and o["person"] <= CHECK["person_share"]
    print(f"{name:42s}{o['calls']:>24.0f}{o['person']:>14.0%}  {'meets' if ok else 'breaks'}")
acc_slow_all = float((slow == yte).mean()); lo, hi = m.wilson(int((slow == yte).sum()), n)
print(f"\nslow tier answering all {n}: accuracy {acc_slow_all:.3f}, Wilson [{lo:.3f}, {hi:.3f}]")

                                             slow-tier calls /1000   to a person  checklist (<=120 calls, <=5% people)
encoder up (normal run)                                         85            0%  meets
encoder down, send all to the slow tier                       1000            0%  breaks
encoder down, send all to a person                               0          100%  breaks

slow tier answering all 200: accuracy 0.950, Wilson [0.910, 0.973]


**What we got.** Sending everything to the slow tier uses 1,000 calls per 1,000 messages against a budget of 120, and sending everything to a person puts 100% of messages in a queue sized for 5%. Neither meets the checklist. The slow tier is the one that still gives answers (95.0%, Wilson interval 91.0% to 97.3%), at about 12 times the usual call volume.

**What this shows for the pattern.** A fallback has a price, and the price is a budget that breaks. Write down, before the outage, which budget breaks first and for how long (for example "the slow-tier budget may be exceeded for up to one hour; after that, messages queue for a person"), because the person deciding under pressure at 3 a.m. should not also be deciding the policy. Each tier needs one declared failure outcome, the system needs a test that every message ends with one, and the checklist needs a degraded-mode line beside the normal-mode targets.

## Part 4. A test that no message is left without an outcome

The check below runs the cascade with fallback at five failure rates, including a total outage, and asserts that every message ends as `encoder`, `llm` or `person` and that the same call twice gives the same outcome. It is the test that would fail if a later change let an exception through.

In [5]:
OK = {"encoder", "llm", "person"}
for p in (0.0, 0.1, 0.5, 0.9, 1.0):
    out = [cascade(i, p, True) for i in range(n)]
    assert all(o[0] in OK for o in out), f"untyped outcome at p={p}"
    assert out == [cascade(i, p, True) for i in range(n)], f"not repeatable at p={p}"
print("every message ends with a typed outcome at five failure rates, and repeating the run gives the same outcomes")

out_json = {"n": n, "acts": int(one.sum()), "escalated": int((~one).sum()),
            "designs": {f"{d}|{p}": rows[(d, p)] for (d, p) in rows},
            "slow_all_acc": acc_slow_all,
            "fallback_options": options, "checklist": CHECK}
(RESULTS / "12_fallback_on_failure.json").write_text(json.dumps(out_json, indent=2, default=float))

every message ends with a typed outcome at five failure rates, and repeating the run gives the same outcomes


2955

**What we got.** The assertions pass at every rate. A cascade is only as good as its worst-handled failure, so keep this test in the continuous-integration run beside the release checklist: it fails the build if a path can end without an outcome.